In [1]:
import kagglehub
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from kagglehub import KaggleDatasetAdapter
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

2026-03-09 18:58:13.470866: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773082693.682967      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773082693.757415      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773082694.235559      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773082694.235622      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773082694.235625      55 computation_placer.cc:177] computation placer alr

In [2]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "akmittal/quotes-dataset",
    "quotes.json",
)   

In [3]:
df = df.drop_duplicates(subset=['Quote'])

In [4]:
df.isnull().sum()

Quote         0
Author        0
Tags          0
Popularity    0
Category      0
dtype: int64

In [5]:
df = df[df["Quote"].apply(lambda x: len(x.split()) < 25)]

In [6]:
def clean_text(text):
    text = re.sub(r'([.!?])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["Quote"] = df["Quote"].apply(clean_text)

In [7]:
texts = df['Quote'].astype(str).values

In [8]:
tokenizer = Tokenizer()

In [9]:
tokenizer.fit_on_texts(df['Quote'])

In [10]:
tokenizer.word_index["<START>"] = len(tokenizer.word_index) + 1
tokenizer.word_index["<END>"] = len(tokenizer.word_index) + 1

In [11]:
vocab_size = len(tokenizer.word_index) + 1

In [12]:
sequences = tokenizer.texts_to_sequences(texts)

In [13]:
MAX_LEN = 25

max_len = min(MAX_LEN, max(len(seq) for seq in sequences))

In [14]:
X = []
y = []

for seq in sequences:
    # Add start and end tokens
    seq = [tokenizer.word_index["<START>"]] + seq + [tokenizer.word_index["<END>"]]

    for i in range(1, len(seq)):
        ngram = seq[:i+1]
        ngram = ngram[-max_len:]  # keep only last max_len tokens

        padded = [0] * (max_len - len(ngram)) + ngram  # left padding

        X.append(padded[:-1])  # input sequence
        y.append(padded[-1])   # target next token

In [15]:
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (344429, 24)
y shape: (344429,)


In [16]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))

# Shuffle before splitting
dataset = dataset.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)

train_size = int(0.9 * len(X))

train_dataset = (
    dataset.take(train_size)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    dataset.skip(train_size)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

I0000 00:00:1773082724.980781      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [17]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

In [18]:
embed_dim = 256
num_heads = 4
ff_dim = 512

def last_token(t):
    return t[:, -1, :]

inputs = Input(shape=(max_len-1,))

# Token embedding
token_embedding = Embedding(vocab_size, embed_dim)
x = token_embedding(inputs)

# Positional embedding
positions = tf.range(start=0, limit=max_len-1, delta=1)
positions = tf.expand_dims(positions, 0)  # shape (1, max_len-1)
pos_embedding = Embedding(max_len, embed_dim)(positions)
x = token_embedding(inputs) + pos_embedding

# Transformer block
attn_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(x, x)
attn_output = Dropout(0.1)(attn_output)

x = Add()([x, attn_output])
x = LayerNormalization()(x)

# Feedforward
ff = Dense(ff_dim, activation="relu")(x)
ff = Dense(embed_dim)(ff)
ff = Dropout(0.1)(ff)

x = Add()([x, ff])
x = LayerNormalization()(x)

# Last token representation
x = Lambda(last_token, output_shape=(embed_dim,))(x)

# Standard softmax output
outputs = Dense(vocab_size, activation="softmax")(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 24)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 24, 256)   │  5,547,776 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 24, 256)   │          0 │ embedding[1][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 24, 256)   │  1,051,904 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 24, 256)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 24, 256)   │          0 │ add[0][0],        │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 24, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 24, 512)   │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24, 256)   │    131,328 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 24, 256)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 24, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 256)   │        512 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 256)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 21671)     │  5,569,447 │ lambda[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,433,063 (47.43 MB)

 Trainable params: 12,433,063 (47.43 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

In [20]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=[early_stop, tf.keras.callbacks.ReduceLROnPlateau(patience=2)]
)

Epoch 1/100


I0000 00:00:1773082731.285050     122 service.cc:152] XLA service 0x7834a8014850 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773082731.285091     122 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773082731.934074     122 cuda_dnn.cc:529] Loaded cuDNN version 91002


  19/4844 ━━━━━━━━━━━━━━━━━━━━ 42s 9ms/step - accuracy: 0.0374 - loss: 9.7787       

I0000 00:00:1773082735.670386     122 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4844/4844 ━━━━━━━━━━━━━━━━━━━━ 57s 10ms/step - accuracy: 0.1262 - loss: 6.3186 - val_accuracy: 0.1838 - val_loss: 5.1882 - learning_rate: 3.0000e-04
Epoch 2/100
4844/4844 ━━━━━━━━━━━━━━━━━━━━ 44s 9ms/step - accuracy: 0.1883 - loss: 5.1571 - val_accuracy: 0.2252 - val_loss: 4.5905 - learning_rate: 3.0000e-04
Epoch 3/100
4844/4844 ━━━━━━━━━━━━━━━━━━━━ 44s 9ms/step - accuracy: 0.2220 - loss: 4.6467 - val_accuracy: 0.2704 - val_loss: 4.0665 - learning_rate: 3.0000e-04
Epoch 4/100
4844/4844 ━━━━━━━━━━━━━━━━━━━━ 45s 9ms/step - accuracy: 0.2622 - loss: 4.1940 - val_accuracy: 0.3168 - val_loss: 3.6151 - learning_rate: 3.0000e-04
Epoch 5/100
4844/4844 ━━━━━━━━━━━━━━━━━━━━ 44s 9ms/step - accuracy: 0.3001 - loss: 3.7736 - val_accuracy: 0.3751 - val_loss: 3.1754 - learning_rate: 3.0000e-04
Epoch 6/100
4844/4844 ━━━━━━━━━━━━━━━━━━━━ 44s 9ms/step - accuracy: 0.3436 - loss: 3.4039 - val_accuracy: 0.4229 - val_loss: 2.8486 - learning_rate: 3.0000e-04
Epoch 7/100
4844/4844 ━━━━━━━━━━━━━━━━━━━━ 44s 9ms/

In [21]:
model.save("next_quote_model.keras")

In [22]:
import json

with open("config.json", "w") as f:
    json.dump({"max_len": max_len}, f)

In [23]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)